In [ ]:

import numpy as np
import pandas as pd
import json
from pathlib import Path



SOURCE_PATH = "./train_balanced.csv"

ARTIFACT_DIR = Path("./digikala_cluster_artifacts/classes")

OUTPUT_PATH = "./digikala_1200k_cluster_sample_rate.csv"

TARGET = 400000
MIN_PER_CLUSTER = 100

SEED = 42

TEXT_COLUMN = "text"
LABEL_COLUMN = "recommendation_status_clean"
RATE_COLUMN = "rate"

classes = [
    "recommended",
    "no_idea",
    "not_recommended"
]

sampled_classes = []


# ----------------------------------
# 2. Process each class
# ----------------------------------

for class_index, class_name in enumerate(classes):

    print("\n" + "=" * 50)
    print("Class:", class_name)
    print("=" * 50)

    # Load class data
    df = pd.read_csv(
        SOURCE_PATH,
        usecols=[
            TEXT_COLUMN,
            LABEL_COLUMN,
            RATE_COLUMN
        ]
    )

    df = df[df[LABEL_COLUMN] == class_name].copy()

    df = df.drop_duplicates(
        subset=[TEXT_COLUMN]
    ).reset_index(drop=True)

    # Load saved cluster IDs
    class_dir = ARTIFACT_DIR / class_name

    cluster_ids = np.load(
        class_dir / "cluster_ids.npy",
        mmap_mode="r"
    )

    # Load metadata
    with open(
        class_dir / "metadata.json",
        "r",
        encoding="utf-8"
    ) as file:
        metadata = json.load(file)

    k = metadata["n_clusters"]

    # Check data consistency
    if len(df) != len(cluster_ids):
        raise ValueError(
            f"Data and cluster IDs do not match for {class_name}!"
        )

    # ----------------------------------
    # 3. Count samples in each cluster
    # ----------------------------------

    cluster_ids_array = np.asarray(
        cluster_ids,
        dtype=np.int32
    )

    if np.any(cluster_ids_array < 0) or np.any(cluster_ids_array >= k):
        raise ValueError(
            f"Invalid cluster IDs for {class_name}!"
        )

    cluster_counts = np.bincount(
        cluster_ids_array,
        minlength=k
    )

    non_empty_clusters = np.where(
        cluster_counts > 0
    )[0]

    if cluster_counts.sum() < TARGET:
        raise ValueError(
            f"Not enough samples in {class_name}!"
        )

    if TARGET < len(non_empty_clusters) * MIN_PER_CLUSTER:
        raise ValueError(
            f"TARGET is too small to provide {MIN_PER_CLUSTER} "
            f"samples to every non-empty cluster in {class_name}!"
        )

    print("Total samples:", len(df))
    print("Total clusters:", k)
    print("Non-empty clusters:", len(non_empty_clusters))

    # ----------------------------------
    # 4. Hybrid quota allocation
    # ----------------------------------

    # Step 1: Minimum quota for each non-empty cluster
    quotas = np.minimum(
        cluster_counts,
        MIN_PER_CLUSTER
    ).astype(np.int64)

    # Step 2: Remaining capacity of each cluster
    remaining_capacity = cluster_counts - quotas

    # Number of samples still needed
    remaining_samples = TARGET - quotas.sum()

    # Step 3: Allocate remaining samples proportionally
    if remaining_samples > 0:

        total_capacity = remaining_capacity.sum()

        if total_capacity < remaining_samples:
            raise ValueError(
                f"Insufficient remaining capacity for {class_name}!"
            )

        exact_quotas = (
            remaining_samples
            * remaining_capacity
            / total_capacity
        )

        additional_quotas = np.floor(
            exact_quotas
        ).astype(np.int64)

        # Distribute leftover samples using largest remainder
        fractions = exact_quotas - additional_quotas

        additional_remaining = (
            remaining_samples - additional_quotas.sum()
        )

        order = np.argsort(
            -fractions,
            kind="stable"
        )

        for cluster_id in order:

            if additional_remaining == 0:
                break

            if additional_quotas[cluster_id] < remaining_capacity[cluster_id]:

                additional_quotas[cluster_id] += 1
                additional_remaining -= 1

        quotas += additional_quotas

    # Check quota allocation
    if quotas.sum() != TARGET:
        raise ValueError(
            f"Quota allocation error for {class_name}!"
        )

    if np.any(quotas > cluster_counts):
        raise ValueError(
            f"Quota exceeds cluster capacity for {class_name}!"
        )

    # ----------------------------------
    # 5. Prepare cluster members
    # ----------------------------------

    sorted_indices = np.argsort(
        cluster_ids_array,
        kind="stable"
    )

    boundaries = np.cumsum(cluster_counts)

    starts = np.concatenate(
        ([0], boundaries[:-1])
    )

    # ----------------------------------
    # 6. Sample from each cluster
    # ----------------------------------

    rng = np.random.default_rng(
        SEED + class_index
    )

    selected_indices = []

    for cluster_id in non_empty_clusters:

        start = starts[cluster_id]
        end = boundaries[cluster_id]

        members = sorted_indices[start:end]

        quota = int(quotas[cluster_id])

        chosen = rng.choice(
            members,
            size=quota,
            replace=False
        )

        selected_indices.extend(chosen)

    selected_indices = np.array(
        selected_indices,
        dtype=np.int64
    )

    # ----------------------------------
    # 7. Create sampled dataframe
    # ----------------------------------

    sampled_df = df.iloc[
        selected_indices
    ].copy()

    sampled_df["cluster_id"] = cluster_ids_array[
        selected_indices
    ]

    sampled_classes.append(sampled_df)

    # ----------------------------------
    # 8. Print sampling report
    # ----------------------------------

    print("\nSelected samples:", len(sampled_df))

    print("\nCluster sampling report:")

    cluster_report = pd.DataFrame({
        "cluster_id": np.arange(k),
        "original_count": cluster_counts,
        "selected_count": quotas
    })

    cluster_report = cluster_report[
        cluster_report["original_count"] > 0
    ].reset_index(drop=True)

    print(cluster_report.to_string(index=False))

    print(
        "\nMinimum selected per non-empty cluster:",
        cluster_report["selected_count"].min()
    )


# ----------------------------------
# 9. Combine all classes
# ----------------------------------

final_df = pd.concat(
    sampled_classes,
    ignore_index=True
)

# Shuffle final dataset
final_df = final_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)


# ----------------------------------
# 10. Final checks
# ----------------------------------

print("\n" + "=" * 50)
print("FINAL DATASET")
print("=" * 50)

print("Total samples:", len(final_df))

print("\nClass distribution:")

print(
    final_df[LABEL_COLUMN].value_counts()
)

print("\nRating distribution:")

print(
    final_df[RATE_COLUMN].value_counts(dropna=False).sort_index()
)

print("\nMissing rates:", final_df[RATE_COLUMN].isna().sum())

print("\nDuplicate texts:", final_df[TEXT_COLUMN].duplicated().sum())

print("\nFinal columns:")
print(final_df.columns.tolist())


# ----------------------------------
# 11. Save dataset
# ----------------------------------

final_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved successfully!")
print("Output:", OUTPUT_PATH)



Class: recommended
Total samples: 2771263
Total clusters: 200
Non-empty clusters: 200

Selected samples: 400000

Cluster sampling report:
 cluster_id  original_count  selected_count
          0           10284            1507
          1           23652            3353
          2           10295            1508
          3           12616            1829
          4           13188            1908
          5           11765            1711
          6           26241            3711
          7            4631             726
          8           19880            2832
          9           12327            1789
         10           26594            3759
         11           26774            3784
         12           17840            2550
         13           21837            3102
         14           12004            1744
         15           20217            2879
         16           19324            2755
         17           18777            2680
         18           194